# Task 6: Evidence-Based Recommendations

This notebook takes the CEO Agent's top recommendations from Task 5
and structures them into the exact format required:

- Recommendation
- Supporting Evidence (3 sources)
- Expected Impact (revenue growth / market differentiation / customer acquisition)
- Risk Assessment (financial / operational / strategic risk)

We reuse Task 4's risks/opportunities/trends as the evidence pool.

In [2]:
import json
import os
import requests
from dotenv import load_dotenv

load_dotenv("../.env")

print("Using Ollama (phi4-mini) as the reasoning engine")

Using Ollama (phi4-mini) as the reasoning engine


## Step 1: Load Task 4 and Task 5 Results

We load the risks/opportunities/trends (evidence pool) and the
CEO's prioritized recommendations from the previous notebooks.

In [3]:
# Load Task 4 results (risks, opportunities, trends)
with open("../data/intelligence_results.json", "r", encoding="utf-8") as f:
    intelligence = json.load(f)

risks = intelligence["risks"]
opportunities = intelligence["opportunities"]
trends = intelligence["trends"]

# Load Task 5 CEO recommendation
with open("../data/ceo_recommendation.txt", "r", encoding="utf-8") as f:
    ceo_recommendation = f.read()

print(f"Risks loaded         : {len(risks)}")
print(f"Opportunities loaded : {len(opportunities)}")
print(f"Trends loaded         : {len(trends)}")
print(f"CEO recommendation loaded : {len(ceo_recommendation)} characters")

Risks loaded         : 4
Opportunities loaded : 4
Trends loaded         : 4
CEO recommendation loaded : 4248 characters


## Step 2: Generate Evidence-Based Recommendations

We ask the LLM to read the CEO's strategic priorities and the underlying
risks/opportunities/trends, then output exactly 3 recommendations in
the structured format the exam requires.

In [6]:
def generate_evidence_based_recommendations(ceo_recommendation, risks, opportunities, trends):

    evidence_pool = "RISKS:\n"
    for r in risks:
        evidence_pool += f"- {r['title']}: {r['evidence']}\n"

    evidence_pool += "\nOPPORTUNITIES:\n"
    for o in opportunities:
        evidence_pool += f"- {o['title']}: {o['evidence']}\n"

    evidence_pool += "\nTRENDS:\n"
    for t in trends:
        evidence_pool += f"- {t['title']}: {t['evidence']}\n"

    prompt = f"""You are a strategic analyst for Lufthansa Group.

CEO'S STRATEGIC PRIORITIES:
{ceo_recommendation}

EVIDENCE POOL:
{evidence_pool}

Based on the CEO's priorities above, create exactly 3 evidence-based recommendations.

STRICT RULES:
- "supporting_evidence" must have EXACTLY 3 items, each a real evidence source from the pool above
- "expected_impact" must rate each of these 3 fixed categories as High/Medium/Low: revenue_growth, market_differentiation, customer_acquisition
- "risk_assessment" must rate each of these 3 fixed categories as High/Medium/Low: financial_risk, operational_risk, strategic_risk

Respond ONLY with valid JSON in this exact format:
[
  {{
    "recommendation": "one clear sentence",
    "supporting_evidence": ["source 1", "source 2", "source 3"],
    "expected_impact": {{"revenue_growth": "High/Medium/Low", "market_differentiation": "High/Medium/Low", "customer_acquisition": "High/Medium/Low"}},
    "risk_assessment": {{"financial_risk": "High/Medium/Low", "operational_risk": "High/Medium/Low", "strategic_risk": "High/Medium/Low"}}
  }}
]"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "phi4-mini",
            "prompt": prompt,
            "stream": False
        }
    )

    text = response.json()["response"]
    text = text.replace("```json", "").replace("```", "")
    start = text.find("[")
    end = text.rfind("]") + 1
    text = text[start:end]

    return json.loads(text)


recommendations = generate_evidence_based_recommendations(ceo_recommendation, risks, opportunities, trends)
print(f"Generated {len(recommendations)} recommendations")

Generated 3 recommendations


In [8]:
for i, rec in enumerate(recommendations, 1):
    print(f"\n=== Recommendation {i} ===")
    print(f"Recommendation    : {rec['recommendation']}")
    print(f"Supporting Evidence:")
    for ev in rec['supporting_evidence']:
        print(f"  - {ev}")
    print(f"Expected Impact   : {rec['expected_impact']}")
    print(f"Risk Assessment   : {rec['risk_assessment']}")


=== Recommendation 1 ===
Recommendation    : Prioritize Fleet Modernization and Sustainability to leverage opportunities in emerging markets.
Supporting Evidence:
  - Expansion into emerging markets like Asia and Africa: 'Stritt um Drehkreuze Emirates und Co. setzen Lufthansa zu.'
  - Fleet modernization for sustainability and efficiency: 'Deutsche Lufthansa AG engages in the provision of passenger, freight...'
  - Sustainability Initiatives with SAF Adoption: Nachhaltigkeit bei Lufthansa Group moderne Flotte Innovationen
Expected Impact   : {'revenue_growth': 'High', 'market_differentiation': 'Medium', 'customer_acquisition': 'Low'}
Risk Assessment   : {'financial_risk': 'Medium', 'operational_risk': 'Low', 'strategic_risk': 'Low'}

=== Recommendation 2 ===
Recommendation    : Enhance Digital Transformation Capabilities to optimize operational efficiency and improve customer service.
Supporting Evidence:
  - Digital transformation through AI and cloud technology integration: 'Lufthan

In [9]:
from datetime import datetime

output = {
    "recommendations": recommendations,
    "generated_at": datetime.now().isoformat()
}

os.makedirs("../data", exist_ok=True)
with open("../data/evidence_based_recommendations.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("Saved evidence-based recommendations to data/evidence_based_recommendations.json")

Saved evidence-based recommendations to data/evidence_based_recommendations.json
